# Geographic-Scale Dataset Build

Builds a place-based dataset for the scaling sweep:

- The top-N (default 10) species at the place are the **target pool**. Each gets its own class folder, downloaded with a large per-species budget.
- All other species present in the area are pooled into `non_target`, with a small per-species download budget *and* a per-species cap when assembling the pool so no single non-top species dominates.
- AudioSet ambient + BirdNET no-bird clips are added to `non_target` for negative diversity.

Top-N species are **never** placed in `non_target` — they only ever appear as targets in `train.ipynb`. All downloads are cached: re-running only fetches what's missing.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import pyrootutils

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator="pyproject.toml",
    pythonpath=True,
    dotenv=True,
)

# Parameters

In [ ]:
from building.geographic_scale import scale_slug

LAT, LON = 48.8566, 2.3522  # PARIS
RADIUS_KM = 50
N_TARGETS = 10

# Asymmetric per-species download budget: top-N get the full budget, the
# rest get a small fixed quota each. download_and_process is idempotent
# and budget-bounded, so it never re-fetches a species already cached at
# >= the requested quota.
TARGET_CLIPS_PER_SPECIES = 5000
NON_TARGET_CLIPS_PER_SPECIES = 250

# Hard cap on each non-top species' contribution to the on-disk non_target
# class. Defends against the case where a non-top species was downloaded
# heavily in another collection and now has many more clips on disk than
# the per-species download quota above.
NON_TARGET_PER_SPECIES_CAP = 300

# Area discovery (matches analysis.ipynb defaults)
MIN_AREA_RECORDINGS = 10
ANALYSIS_MAX_BIRDNET = 500
ANALYSIS_MIN_CONF = 0.85

# AudioSet ambient/anthropic noise (diversified across ~60 FOCUS_CLASSES)
AS_CLIPS_PER_CLASS = 250
AS_MAX_TOTAL_CLIPS = 15000

# Dataset folder slug encodes (n_targets, place) so the experiment context
# is visible at the filesystem level and aligned with the results folder.
COLLECTION_NAME = scale_slug(LAT, LON, RADIUS_KM, n_targets=N_TARGETS)
print(f"collection: {COLLECTION_NAME}")

collection: scale_s10_49_2_r50


# Discover area species and pick top-N as targets

`combined_species_table` returns the area's species sorted by evidence
(XC presence + BirdNET detections). The top-N become target classes; the
rest are pooled into `non_target`.

In [4]:
from building.analysis import (
    area_audio_cache_dir,
    bbox_from_radius,
    birdnet_area_analysis,
    combined_species_table,
    fetch_area_recordings,
    recordings_to_df,
)

bbox = bbox_from_radius(LAT, LON, RADIUS_KM)
area_df = recordings_to_df(await fetch_area_recordings(bbox))
bn = await birdnet_area_analysis(
    area_df,
    max_recordings=ANALYSIS_MAX_BIRDNET,
    min_confidence=ANALYSIS_MIN_CONF,
    cache_dir=area_audio_cache_dir(LAT, LON, RADIUS_KM),
)
combined = combined_species_table(
    area_df, bn["detections"], min_recordings=MIN_AREA_RECORDINGS
)
all_species = combined["scientific_name"].tolist()
TARGET_SPECIES = all_species[:N_TARGETS]
OTHER_SPECIES = all_species[N_TARGETS:]
print(f"Top-{N_TARGETS} target species:")
for sp in TARGET_SPECIES:
    print(f"  {sp}")
print(f"\n{len(OTHER_SPECIES)} other species pooled into non_target")

BirdNET analyse:   0%|          | 0/500 [00:00<?, ?it/s]

Top-10 target species:
  Hippolais polyglotta
  Psittacula krameri
  Parus major
  Anthus spinoletta
  Dendrocoptes medius
  Curruca communis
  Fringilla coelebs
  Sylvia atricapilla
  Regulus ignicapilla
  Certhia brachydactyla

28 other species pooled into non_target


# Stream-download target species (heavy budget)

In [ ]:
from building.data.download import download_and_process
from building.data.sources import XCSource, EBirdSource

with XCSource() as xc, EBirdSource() as eb:
    for sp in TARGET_SPECIES:
        await download_and_process(sp, TARGET_CLIPS_PER_SPECIES, [xc, eb])


=== [Hippolais polyglotta] target=10000 clips (5000/source × 2 sources)  on disk=7759 [xc=3524/5000, ebird=4235/5000] ===
[Hippolais polyglotta] --- Phase A: per-source quota (parallel) ---
[Hippolais polyglotta / xc] 3524/5000 on disk; queued 401 of 401 available.


[Hippolais polyglotta/xc] recs:   0%|          | 0/401 [00:00<?, ?it/s]

[Hippolais polyglotta/xc] clips:  70%|#######   | 3524/5000 [00:00<?, ?it/s]

[Hippolais polyglotta / ebird] 4235/5000 on disk; queued 765 of 1306 available.


[Hippolais polyglotta/ebird] recs:   0%|          | 0/765 [00:00<?, ?it/s]

...

[Psittacula krameri/xc] recs:   0%|          | 0/310 [00:00<?, ?it/s]

[Psittacula krameri/xc] clips:   9%|9         | 452/5000 [00:00<?, ?it/s]

[Psittacula krameri / ebird] 148/5000 on disk; queued 3967 of 3967 available.


[Psittacula krameri/ebird] recs:   0%|          | 0/3967 [00:00<?, ?it/s]

[Psittacula krameri/ebird] clips:   3%|2         | 148/5000 [00:00<?, ?it/s]

# Stream-download other species (light budget, pooled as non-target)

In [ ]:
with XCSource() as xc, EBirdSource() as eb:
    for sp in OTHER_SPECIES:
        await download_and_process(sp, NON_TARGET_CLIPS_PER_SPECIES, [xc, eb])

# Stream-download AudioSet ambient clips

In [ ]:
from building.data.audioset import AudioSetConfig, stream_download_audioset_async

as_cfg = AudioSetConfig(
    clips_per_class=AS_CLIPS_PER_CLASS,
    max_total_clips=AS_MAX_TOTAL_CLIPS,
)
await stream_download_audioset_async(as_cfg)

# Pre-split summary

In [ ]:
from building.data.dataset import AUDIOSET_DIR, BIRDNET_NO_BIRD_DIR, SUBSAMPLES_DIR


def _count(folder):
    return len(list(folder.glob("*.wav"))) if folder.exists() else 0


print("Target classes:")
for sp in TARGET_SPECIES:
    print(f"  {sp}: {_count(SUBSAMPLES_DIR / sp.replace(' ', '_'))}")

as_total, as_classes = 0, 0
if AUDIOSET_DIR.exists():
    for d in sorted(AUDIOSET_DIR.iterdir()):
        if d.is_dir():
            n = _count(d)
            if n:
                as_total += n
                as_classes += 1
print(f"\nAudioSet: {as_total} clips across {as_classes} classes")

other_total, other_species_present, capped_total = 0, 0, 0
for sp in OTHER_SPECIES:
    n = _count(SUBSAMPLES_DIR / sp.replace(" ", "_"))
    if n:
        other_total += n
        other_species_present += 1
        capped_total += min(n, NON_TARGET_PER_SPECIES_CAP)
print(
    f"\nOther birds: {other_total} clips on disk "
    f"across {other_species_present}/{len(OTHER_SPECIES)} species"
)
print(
    f"After per-species cap ({NON_TARGET_PER_SPECIES_CAP}): "
    f"{capped_total} clips eligible for non_target xc_other pool"
)

print(f"\nBirdNET no-bird clips: {_count(BIRDNET_NO_BIRD_DIR)}")

# Per-source RMS distribution (pre-split)

Looking at raw clip levels before assembling the dataset so we can see
the level imbalance per source. Bins of 2 dB from -70 to 0 dBFS, chosen
from an empirical sample.

In [ ]:
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from building.data.dataset import AUDIOSET_DIR, BIRDNET_NO_BIRD_DIR, SUBSAMPLES_DIR


def _rms_dbfs(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float32)
    rms = np.sqrt(np.mean(x**2))
    return float(20.0 * np.log10(max(rms, 1e-10)))


def _scan(files: list) -> np.ndarray:
    out: list[float] = []
    for f in tqdm(files, desc=f"scan ({len(files)})", leave=False):
        try:
            x, _ = sf.read(str(f), dtype="float32", always_2d=False)
            if x.ndim > 1:
                x = x.mean(axis=-1)
            out.append(_rms_dbfs(x))
        except Exception:
            pass
    return np.array(out)


groups: list[tuple[str, np.ndarray]] = []

# One panel per target species.
for sp in TARGET_SPECIES:
    folder = SUBSAMPLES_DIR / sp.replace(" ", "_")
    if folder.exists():
        groups.append((sp, _scan(sorted(folder.glob("*.wav")))))

# Other birds aggregated.
other_files: list = []
for sp in OTHER_SPECIES:
    folder = SUBSAMPLES_DIR / sp.replace(" ", "_")
    if folder.exists():
        other_files.extend(folder.glob("*.wav"))
groups.append((f"Other birds ({len(OTHER_SPECIES)} sp.)", _scan(other_files)))

# BirdNET no-bird clips.
if BIRDNET_NO_BIRD_DIR.exists():
    groups.append(
        ("BirdNET no-bird", _scan(sorted(BIRDNET_NO_BIRD_DIR.glob("*.wav"))))
    )

# AudioSet aggregated across focus classes.
if AUDIOSET_DIR.exists():
    as_files: list = []
    for d in sorted(AUDIOSET_DIR.iterdir()):
        if d.is_dir():
            as_files.extend(d.glob("*.wav"))
    groups.append(("AudioSet (all)", _scan(as_files)))

print(
    f"{'group':<32s} {'n':>5s}  {'q05':>6s} {'q50':>6s} {'q95':>6s}  "
    f"{'min':>6s} {'max':>6s}"
)
for label, a in groups:
    if len(a) == 0:
        continue
    print(
        f"{label:<32s} {len(a):>5d}  "
        f"{np.quantile(a, 0.05):>+6.1f} {np.median(a):>+6.1f} {np.quantile(a, 0.95):>+6.1f}  "
        f"{a.min():>+6.1f} {a.max():>+6.1f}"
    )

bins = np.arange(-70, 1, 2)
n = len(groups)
fig, axes = plt.subplots(n, 1, figsize=(10, max(2.0, 1.6 * n)), sharex=True)
if n == 1:
    axes = [axes]
for ax, (label, a) in zip(axes, groups):
    if len(a):
        ax.hist(a, bins=bins, color="steelblue", edgecolor="black", alpha=0.85)
        ax.axvline(
            np.median(a), color="red", lw=1, ls="--", label=f"median {np.median(a):+.1f}"
        )
        ax.legend(loc="upper left", fontsize=8, frameon=False)
    ax.set_ylabel(label, rotation=0, ha="right", va="center")
axes[-1].set_xlabel("RMS (dBFS)")
fig.suptitle("Per-source RMS distribution (all data, pre-split)")
plt.tight_layout()
plt.show()

# Assemble train/val/test

Each top-N species becomes its own class folder; `non_target` is the
balanced pool of (audioset + xc_other + birdnet_no_bird), with each
non-top species capped at `NON_TARGET_PER_SPECIES_CAP` clips before
pooling. Top-N species are never passed as non-target species, so they
cannot leak into `non_target`.

In [ ]:
from building.data.dataset import build_task_dataset

dataset_path = build_task_dataset(
    COLLECTION_NAME,
    target_species=TARGET_SPECIES,
    non_target_species=OTHER_SPECIES,
    non_target_per_species_cap=NON_TARGET_PER_SPECIES_CAP,
)
print(dataset_path)